# Setting up a new MacBook for development

Work through the sections in order. Every command is run in **Terminal** (or the VS Code integrated terminal) unless stated otherwise.
Written for Apple Silicon Macs on macOS with the default `zsh` shell; Intel differences are called out.

| # | Section | Result |
|---|---------|--------|
| 1 | Homebrew | `brew` package manager |
| 2 | uv | fast Python package / project / version manager |
| 3 | Python 3.12.10 | interpreter managed by uv |
| 4 | PATH | know where it is defined and how to change it |
| 5 | CLI tools | `aws`, `kubectl`, `terraform`, ... |
| 6 | VS Code | editor, extensions, shortcuts |

## 1. Install Homebrew

Homebrew is the package manager everything else is installed through. The installer also installs the **Xcode Command Line Tools** (git, clang, make) if they are missing, which can take a few minutes.

In [ ]:
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

**Add `brew` to your PATH.** The installer prints these "Next steps" at the end; run them:

In [ ]:
# Apple Silicon: Homebrew lives in /opt/homebrew. (Intel Macs: /usr/local, already on PATH, nothing to add)
echo 'eval "$(/opt/homebrew/bin/brew shellenv)"' >> ~/.zprofile
eval "$(/opt/homebrew/bin/brew shellenv)"

# Verify:
brew --version
brew doctor        # should end with "Your system is ready to brew."


Everyday commands:

| Command | Purpose |
|---------|---------|
| `brew install <formula>` | install a command-line tool |
| `brew install --cask <app>` | install a GUI app (e.g. `visual-studio-code`) |
| `brew update && brew upgrade` | refresh the index and upgrade everything |
| `brew list` | show installed packages |
| `brew uninstall <name>` | remove a package |
| `brew search <text>` | search formulae and casks |

## 2. Install uv
[uv](https://docs.astral.sh/uv/) replaces `pip`, `pipenv`, `virtualenv`,`pyenv` and `poetry` with a single tool.


In [ ]:
brew install uv
uv --version
# Optional: shell completion.
echo 'eval "$(uv generate-shell-completion zsh)"' >> ~/.zshrc


## 3. Install Python 3.12.10 with uv

macOS ships a system `python3` that you should not modify. Let uv download and manage its own interpreter instead.

In [ ]:
uv python install 3.12.10
uv python list --only-installed     # confirm it is there

uv stores interpreters under `~/.local/share/uv/python/`, **not** on your `PATH`, so plain `python` will not point to it. Use it in one of these ways:


In [ ]:
# a) One-off script
uv run --python 3.12.10 python --version

# b) Per-project (recommended): pins the version in .python-version
cd my-project
uv init --python 3.12.10        # new project: pyproject.toml + .python-version
uv add requests                 # adds dependency and creates .venv
uv run python main.py           # runs inside the project's .venv
uv sync                         # recreate .venv from pyproject.toml / uv.lock (new machine, fresh clone)

# c) Make `python` / `python3` available globally (preview feature)
uv python install 3.12.10 --default --preview
# puts shims in ~/.local/bin, which must be on PATH (see section 4)


| Task | Command |
|------|---------|
| Create a venv in the current folder | `uv venv --python 3.12.10` then `source .venv/bin/activate` |
| Pin a project's Python version | `uv python pin 3.12.10` |
| Install a standalone CLI app | `uv tool install ruff` |
| Install from a requirements file | `uv pip install -r requirements.txt` |

> This repo is a uv workspace (see the root `pyproject.toml`): run `uv sync` in the repo root after cloning.

## 4. Managing the PATH variable

`PATH` is a colon-separated list of directories. When you type a command, the shell searches them **left to right** and runs the first match.

### Inspect it
tr means 'translate'; below translates `:` to `\n`

In [ ]:
echo $PATH | tr ':' '\n'      # one directory per line

```text
/usr/local/bin
/Users/yu-shenglee/.local/bin
/opt/homebrew/bin
/opt/homebrew/sbin
/System/Cryptexes/App/usr/bin
/usr/bin
/bin
/usr/sbin
/sbin

```

In [ ]:
which python3                  # which executable wins
which -a python3               # every match, in search order
type -a brew                   # also reveals aliases and functions

### Where it comes from (in the order macOS builds it)

| Source | Scope | Notes |
|--------|-------|-------|
| `/etc/paths` | all users | one directory per line, edited with sudo |
| `/etc/paths.d/*` | all users | one file per tool (e.g. Docker, Wireshark add one) |
| `/usr/libexec/path_helper` | all users | run by `/etc/zprofile`; merges the two sources above |
| `~/.zshenv` | you, **every** zsh | avoid; also affects scripts and non-interactive shells |
| `~/.zprofile` | you, login shells | good for one-time setup such as `brew shellenv` |
| `~/.zshrc` | you, interactive shells | **the usual place for your own `export PATH=...`** |

Terminal.app and the VS Code terminal start login shells, so both `~/.zprofile` and `~/.zshrc` are read.



Add a line **at the end**. Put the new directory first to take priority, last to be a fallback:

In [ ]:
# open the config file (creates it if missing):
code ~/.zshrc        # or: open -e ~/.zshrc  /  nano ~/.zshrc

# prepend (takes priority)
export PATH="$HOME/.local/bin:$PATH"

# append (fallback)
export PATH="$PATH:/some/other/bin"

# Then apply it to the current terminal without restarting:
source ~/.zshrc


### Tips and pitfalls

* Always keep `:$PATH` (or `$PATH:`) in the value. `export PATH="/new/dir"` **replaces** the whole PATH and breaks every command.
* Avoid duplicates from repeated `source`: add `typeset -U path PATH` once near the top of `~/.zshrc`.
* `uv tool update-shell` adds `~/.local/bin` to your shell config for you.
* GUI apps launched from Finder do not read `~/.zshrc`; they use the launchd environment. Start them from the terminal (e.g. `code .`) if they need your PATH.
* System-wide change (rarely needed): `sudo sh -c 'echo /new/dir >> /etc/paths.d/mytool'`.
* Debug a "command not found": `echo $PATH`, check the binary exists (`ls /opt/homebrew/bin`), then open a **new** terminal.

## 5. Install other command-line tools

In [ ]:
# Basics
brew install git gh jq tree wget

# AWS CLI v2
brew install awscli
aws --version
aws configure                       # access key, secret, region
# or SSO:  aws configure sso

# Kubernetes
brew install kubernetes-cli         # provides `kubectl`
kubectl version --client
brew install helm k9s               # optional: package manager and terminal UI

# Terraform (HashiCorp's own tap; homebrew-core no longer ships newer releases)
brew tap hashicorp/tap
brew install hashicorp/tap/terraform
terraform -version
# Open-source alternative: brew install opentofu

# Google Cloud (used by the Terraform GCP exercises in knowledge-base/terraform)
brew install --cask google-cloud-sdk
gcloud auth application-default login


# Enable tab completion for `kubectl` and `terraform` by adding to `~/.zshrc`:
autoload -Uz compinit && compinit
source <(kubectl completion zsh)
complete -o nospace -C /opt/homebrew/bin/terraform terraform



## 6. Set up VS Code

### Install


In [ ]:
brew install --cask visual-studio-code

### Put `code` on your PATH

The cask normally links `code` for you; check with `which code`. If it is missing:

1. Open Visual Studio Code
2. Open the command palette with `Cmd + Shift + P` (or `F1`)
3. Type `Shell Command: Install 'code' command in PATH` and select it
4. Open a new terminal. `code .` now opens the current folder in VS Code

### Install extensions

From the Extensions view (`Cmd + Shift + X`) or the terminal:


In [ ]:
code --install-extension ms-python.python           # Python
code --install-extension ms-python.vscode-pylance   # Pylance (IntelliSense)
code --install-extension ms-toolsai.jupyter         # Jupyter Notebook
code --install-extension charliermarsh.ruff         # lint + format
code --install-extension hashicorp.terraform        # Terraform
code --install-extension ms-kubernetes-tools.vscode-kubernetes-tools

### Python and virtual environments

1. Create the environment with uv (section 3): `uv venv --python 3.12.10` or `uv sync`
2. `Cmd + Shift + P` > **Python: Select Interpreter** > choose `./.venv/bin/python`
3. For notebooks, pick the same `.venv` as the kernel (top right of the notebook)
4. Jupyter in a uv project: `uv add --dev ipykernel`
5. Test in the terminal: `python --version`. Enter `python3` for an interpreter, `exit()` to leave it before installing packages. A `SyntaxError` when running `pip install` usually means you are still inside the interpreter.

### Formatting

Ruff (installed above) replaces autopep8. Enable format-on-save in `settings.json`
(`Cmd + Shift + P` > **Preferences: Open User Settings (JSON)**):

```json
{
  "editor.formatOnSave": true,
  "[python]": { "editor.defaultFormatter": "charliermarsh.ruff" }
}
```

Prefer autopep8 instead? `uv tool install autopep8`, then select it under Settings > Python > Formatting: Provider.

### Key shortcuts

| Go to | Shortcut |
|-------|----------|
| Command palette | `Cmd + Shift + P` |
| Search across files | `Cmd + Shift + F` |
| Source control | `Ctrl + Shift + G` |
| Debugger view | `Cmd + Shift + D` |
| Extensions view | `Cmd + Shift + X` |
| Toggle terminal | `` Ctrl + ` `` |
| Zen mode | `Cmd + K`, then `Z` |
| Keyboard shortcuts list | `Cmd + K`, then `Cmd + S` |
| Clear terminal | `Cmd + K` (with terminal focused) |

| Run code | Shortcut |
|----------|----------|
| Start debugging | `F5` |
| Step into | `F11` |
| Run selection/line in Python terminal (REPL) | `Shift + Enter` |
| Trigger IntelliSense | `Ctrl + Space` |

> The REPL (Read-Eval-Print Loop) is provided by the Python extension and lets you work interactively with Python from inside VS Code.